## Step 1 - Load Data

In [17]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern

out = np.load('disc-benchmark-files/training-val-test-data.npz')
th, u = out['th'], out['u']
N=len(th)
n_train = int(0.7 * N)
n_val = int(0.15 * N)
n_test = N - n_train - n_val

# Splitt the data 

u_train, u_val, u_test = u[:n_train], u[n_train:n_train+n_val], u[n_train+n_val:]
th_train, th_val, th_test = th[:n_train], th[n_train:n_train+n_val], th[n_train+n_val:]

print(n_train, n_val, n_test)
print(out.files)


24500 5250 5250
['u', 'th']


## Step 2 -Build NARX model

In [18]:
def create_IO_data(u, y, na, nb):
    """
    Create input-output data for NARX model
    """
    X,Y =[],[]
    for k in range(max(na,nb), len(y)):
        X.append(np.concatenate([u[k-nb:k], y[k-na:k]]))
        Y.append(y[k])
    return np.array(X), np.array(Y)

## Step 3 - Tune na,nb via validation RMSE


In [21]:
for na in [2, 3, 5]:
      for nb in [2, 3, 5]:
          X_tr, Y_tr = create_IO_data(u_train, th_train, na, nb)
          X_v,  Y_v  = create_IO_data(u_val,   th_val,   na, nb)

          # subsample
          idx = np.random.choice(len(X_tr), min(1500, len(X_tr)), replace=False)
          X_sub, Y_sub = X_tr[idx], Y_tr[idx]

          # retrain for this na, nb
          kernel = Matern(nu=2.5) + WhiteKernel(noise_level=0.01)
          gp_tmp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=3, normalize_y=True)
          gp_tmp.fit(X_sub, Y_sub)

          rmse = np.sqrt(np.mean((gp_tmp.predict(X_v) - Y_v)**2))
          print(f'na={na}, nb={nb} -> RMSE={rmse:.5f} rad')

na=2, nb=2 -> RMSE=0.00679 rad
na=2, nb=3 -> RMSE=0.00659 rad
na=2, nb=5 -> RMSE=0.00603 rad
na=3, nb=2 -> RMSE=0.00708 rad
na=3, nb=3 -> RMSE=0.00519 rad
na=3, nb=5 -> RMSE=0.00524 rad
na=5, nb=2 -> RMSE=0.00544 rad
na=5, nb=3 -> RMSE=0.00636 rad
na=5, nb=5 -> RMSE=0.00561 rad


## Step 4 - Train Final GP

In [ ]:
na, nb = 3, 3

X_train, Y_train = create_IO_data(u_train, th_train, na, nb)
X_val,   Y_val   = create_IO_data(u_val,   th_val,   na, nb)
X_test_io, Y_test_io = create_IO_data(u_test, th_test, na, nb)

idx = np.random.choice(len(X_train), 5000, replace=False)

kernel   = Matern(nu=2.5, length_scale=1.0) + WhiteKernel(noise_level=0.01)
gp_final = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=True)
gp_final.fit(X_train[idx], Y_train[idx])

print('Optimized kernel:', gp_final.kernel_)

Optimized kernel: Matern(length_scale=5.11, nu=2.5) + WhiteKernel(noise_level=5.13e-05)


In [ ]:
# Evaluate prediction performance
for name, X, Y in [('Train', X_train, Y_train),
                    ('Val',   X_val,   Y_val),
                    ('Test',  X_test_io, Y_test_io)]:
    pred = gp_final.predict(X)
    rmse = np.sqrt(np.mean((pred - Y)**2))
    nrms = rmse / Y.std() * 100
    print(f'{name} RMSE: {rmse:.5f} rad ({rmse/(2*np.pi)*360:.3f} deg) | NRMS: {nrms:.2f}%')

# Simulation evaluation
def simulation_IO_model(f, ulist, ylist, na, nb, skip=50):
    upast = ulist[skip-nb:skip].tolist()
    ypast = ylist[skip-na:skip].tolist()
    Y = ylist[:skip].tolist()
    for u_k in ulist[skip:]:
        x = np.concatenate([upast, ypast])[None, :]
        ypred = f(x)
        Y.append(ypred)
        upast.append(u_k); upast.pop(0)
        ypast.append(ypred); ypast.pop(0)
    return np.array(Y)

f_gp = lambda x: gp_final.predict(x)[0]

th_train_sim = simulation_IO_model(f_gp, u_train, th_train, na, nb, skip=max(na, nb))
skip = max(na, nb)
rmse_sim = np.sqrt(np.mean((th_train_sim[skip:] - th_train[skip:])**2))
nrms_sim  = rmse_sim / th_train[skip:].std() * 100
print(f'\nTrain simulation RMSE: {rmse_sim:.5f} rad ({rmse_sim/(2*np.pi)*360:.3f} deg) | NRMS: {nrms_sim:.2f}%')

In [ ]:
import matplotlib.pyplot as plt

# Test simulation
th_test_sim = simulation_IO_model(f_gp, u_test, th_test, na, nb, skip=max(na, nb))
skip = max(na, nb)
rmse_test_sim = np.sqrt(np.mean((th_test_sim[skip:] - th_test[skip:])**2))
nrms_test_sim = rmse_test_sim / th_test[skip:].std() * 100
print(f'Test simulation RMSE: {rmse_test_sim:.5f} rad ({rmse_test_sim/(2*np.pi)*360:.3f} deg) | NRMS: {nrms_test_sim:.2f}%')

# Plot prediction vs ground truth (first 500 steps of test set)
pred_test = gp_final.predict(X_test_io)
t = np.arange(500)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(t, Y_test_io[:500], label='True')
axes[0].plot(t, pred_test[:500], '--', label='GP Prediction')
axes[0].set_ylabel('Angle (rad)')
axes[0].set_title('One-step-ahead prediction on test set')
axes[0].legend()

axes[1].plot(t, th_test[skip:500+skip], label='True')
axes[1].plot(t, th_test_sim[skip:500+skip], '--', label='GP Simulation')
axes[1].set_ylabel('Angle (rad)')
axes[1].set_xlabel('Time step')
axes[1].set_title('Simulation on test set')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# GENERATE PREDICTION SUBMISSION
data = np.load('disc-benchmark-files/hidden-test-prediction-submission-file.npz')
upast_test  = data['upast']
thpast_test = data['thpast']

X_pred_test = np.concatenate([upast_test[:, 15-nb:], thpast_test[:, 15-na:]], axis=1)
th_pred = gp_final.predict(X_pred_test)

np.savez('gp-prediction-submission.npz',
           upast=upast_test, thpast=thpast_test, thnow=th_pred)
print('Prediction submission saved.')

# GENERATE SIMULATION SUBMISSION
data = np.load('disc-benchmark-files/hidden-test-simulation-submission-file.npz')
u_hidden  = data['u']
th_hidden = data['th']

th_sim = simulation_IO_model(f_gp, u_hidden, th_hidden, na, nb, skip=50)

assert len(th_sim) == len(th_hidden)
np.savez('gp-simulation-submission.npz', th=th_sim, u=u_hidden)
print('Simulation submission saved.')